# A large region on the GPU: the Amundsen Sea Embayment

The GPU is worth it for **large** maps. Rubble Glacier (the study area in `rubble_glacier_workflow.ipynb`) is about 200 × 200 km; here we map the bed under Thwaites and Pine Island glaciers, a 650 × 700 km box with 289,000 grounded cells at 1 km.

This notebook

1. prepares the region's radar data the same way as the Rubble Glacier workflow,
2. kriges every grounded cell and simulates 16 SGS realizations on the GPU, and
3. compares with the CPU runs on UF's HiPerGator, where GStatSim took up to 9.7 hours for a single map.

Data: Bedmap3 radar bed picks and exposed rock, with the BedMachine Antarctica v3 mask, prepared by `figures/prep_map_data.py` into `figures/data/study_area_ase.npz` (EPSG:3031).

In [ ]:
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "benchmarks"))
from bench_rubble_sgs import region_workflow

from gstatsim_accel import gpu_status, kriging, simulation

status = gpu_status()
print("GPU:", status.device_name if status.available else "not available - " + status.reason)

## 1. Prepare the data

`region_workflow` grids the radar data at 1 km, removes a smooth trend, normal-score transforms the residual and fits an exponential variogram, the same steps as the Rubble Glacier workflow. Two steps are adapted to the region's size: GStatSim's `rbf_trend` would build a 92,000 × 92,000 matrix, so the trend is fitted on a 10 km grid and interpolated; and the variogram is fitted on a random 10,000-point sample.

In [ ]:
t = time.perf_counter()
w = region_workflow("ase", 1000)
df, pred, land, vario = w["df_grid"], w["pred_xy"], w["land"], w["vario"]
nodes = pred[land]
print(f"{len(df):,} gridded data, {len(nodes):,} grounded cells to estimate "
      f"(prepared in {time.perf_counter() - t:.0f} s)")
print("variogram [azimuth, nugget, major, minor, sill, model]:",
      [round(v, 3) if isinstance(v, float) else v for v in vario])

KW = dict(num_points=100, radius=50_000, zz="NRes", backend="gpu")

## 2. Kriging every grounded cell

In [ ]:
t = time.perf_counter()
est, var = kriging.predict(df, nodes, vario, **KW)
t_krige = time.perf_counter() - t
print(f"Kriged {len(nodes):,} cells in {t_krige:.0f} s")

def to_map(values_at_nodes):
    """Back-transform, add the trend, and put the grounded cells on the full grid."""
    bed = np.full(len(pred), np.nan)
    bed[land] = w["nst"].inverse_transform(values_at_nodes.reshape(-1, 1)).ravel() + w["trend"].ravel()[land]
    return bed.reshape(w["rows"], w["cols"])

extent = [pred[:, 0].min() / 1e3, pred[:, 0].max() / 1e3, pred[:, 1].min() / 1e3, pred[:, 1].max() / 1e3]
fig, ax = plt.subplots(figsize=(7, 7))
im = ax.imshow(to_map(est), extent=extent, origin="lower", cmap="gist_earth", vmin=-2000, vmax=1500)
ax.set_title(f"Kriged bed, Amundsen Sea Embayment, 1 km ({t_krige:.0f} s on the GPU)")
ax.set_xlabel("x [km]"); ax.set_ylabel("y [km]")
fig.colorbar(im, ax=ax, shrink=0.8, label="Bed elevation [m]")
plt.show()

## 3. Sixteen SGS realizations

Each realization keeps GStatSim's exact algorithm: cells are simulated one at a time in a random order, each conditioned on everything simulated before it. The GPU runs a whole realization in one thread block and runs the 16 realizations side by side, so 16 take about as long as one.

In [ ]:
n_real = 16
t = time.perf_counter()
sims = simulation.realizations(df, nodes, vario, seeds=list(range(n_real)), **KW)
t_sgs = time.perf_counter() - t
print(f"{n_real} realizations of {len(nodes):,} cells in {t_sgs / 60:.1f} min ({t_sgs / n_real:.0f} s each)")

beds = np.stack([to_map(s) for s in sims])
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, img, title, cmap, lim in (
        (axes[0], beds[0], "Realization 1", "gist_earth", (-2000, 1500)),
        (axes[1], beds[1], "Realization 2", "gist_earth", (-2000, 1500)),
        (axes[2], beds.std(axis=0), f"Spread of {n_real} realizations (std. dev.)", "magma", (0, 400))):
    im = ax.imshow(img, extent=extent, origin="lower", cmap=cmap, vmin=lim[0], vmax=lim[1])
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=ax, shrink=0.7, label="m")
plt.show()

The spread is small along radar lines and large in the gaps between them, which is where the bed is uncertain.

## 4. GPU vs CPU on HiPerGator

A full GStatSim run on this region takes hours, so it was run once on HiPerGator as batch jobs (`scripts/hpg/region_*.sbatch`, `benchmarks/bench_region.py`). The records are in `docs/hpg/`. Each job ran the same inputs as above at 1 km and at 500 m (1.15 million cells).

In [ ]:
steps, cells, hardware = {1000: {}, 500: {}}, {}, {"cpu": set(), "gpu": set()}
for f in sorted((ROOT / "docs" / "hpg").glob("region_ase_*.json")):
    r = json.loads(f.read_text(encoding="utf-8"))
    res = int(r["res_m"])
    steps[res] |= r["steps"]
    cells[res] = r["n_nodes"]
    hardware[r["part"]].add(r["environment"]["cpu_model"] + (
        " + " + r["environment"]["cupy_gpu"]["device_name"] if r["part"] == "gpu" else ""))

def fmt(s):
    return f"{s / 3600:.1f} h" if s >= 3600 else f"{s / 60:.1f} min" if s >= 60 else f"{s:.0f} s"

rows = []
for res, s in steps.items():
    rows.append([f"Kriging, {res} m", f"{cells[res]:,}", fmt(s["gstatsim_kriging"]["seconds"]),
                 fmt(s["gpu_kriging"]["seconds"]),
                 f"{s['gstatsim_kriging']['seconds'] / s['gpu_kriging']['seconds']:.0f}x"])
    many = max((k for k in s if k.startswith("gpu_sgs_x") and k != "gpu_sgs_x1"),
               key=lambda k: s[k]["realizations"])
    per = s[many]["seconds_per_realization"]
    rows.append([f"SGS, {res} m (GPU: {s[many]['realizations']} at once)", f"{cells[res]:,}",
                 fmt(s["gstatsim_sgs_x1"]["seconds"]), fmt(per) + " each",
                 f"{s['gstatsim_sgs_x1']['seconds'] / per:.0f}x"])
pd.DataFrame(rows, columns=["Task", "Cells", "GStatSim (CPU)", "GPU", "Speedup"])

In [ ]:
for part, hw in hardware.items():
    print(f"{part.upper()} jobs: {'; '.join(sorted(hw))}")

**Reading the table.** GStatSim ran on HiPerGator CPU nodes (4 cores; it uses one), the GPU runs on an NVIDIA L4. The CPU nodes are an older AMD EPYC generation than the GPU node, which flatters the GPU somewhat; on the GPU node's own CPU, the Rubble Glacier runs were 62x (kriging) and 131x (SGS) faster on the GPU.

The speedup grows with the size of the map: at 1 km the GPU kriges in seconds what GStatSim does in under an hour; at 500 m, a map that took GStatSim 4.7 hours takes under 2 minutes, and a single SGS realization drops from 9.7 hours to about 4 minutes when 32 are simulated together.